In [1]:
# upload the model
from google.colab import files
uploaded = files.upload()

Saving gesture_classifier.h5 to gesture_classifier.h5
Saving latest_sequence.npy to latest_sequence.npy


In [12]:
!pip install flask flask-ngrok
!pip install flask flask-ngrok pyngrok
!pip install flask pyngrok

In [16]:
import numpy as np
from flask import Flask, request, jsonify
import json
from datetime import datetime
from tensorflow.keras.models import load_model
from pyngrok import ngrok

# --- REPLACE THIS AFTER ROTATING YOUR NGROK TOKEN ---
ngrok.set_auth_token("YOUR_REAL_TOKEN")
# -------------------------------------------------------

GESTURES = ["wave", "peace sign", "fist"]
PASSWORD_ORDER = [0, 1, 2]
T = 30

app = Flask(__name__)

try:
    model = load_model("gesture_classifier.h5", compile=False)
    print("✅ Model loaded")
except Exception as e:
    print("❌ Could not load model:", e)
    model = None


@app.route("/sequence", methods=["POST"])
def receive_sequence():
    data = request.get_json()
    if not data or "sequence" not in data:
        return jsonify({"error": "Missing 'sequence' in request"}), 400

    current_sequence = data["sequence"]

    if model is None:
        return jsonify({"error": "Model not loaded"}), 500

    landmarks = data.get("landmarks", None)
    if landmarks is None:
        return jsonify({"error": "Missing 'landmarks' in request"}), 400

    seq_data = np.array(landmarks, dtype="float32")

    if seq_data.ndim != 2 or seq_data.shape[1] != 63:
        return jsonify({
            "error": f"Invalid landmarks shape {seq_data.shape}, expected (T, 63)"
        }), 400

    if len(seq_data) >= T:
        seq_data = seq_data[:T]
    else:
        pad = np.repeat(seq_data[-1:], T - len(seq_data), axis=0)
        seq_data = np.concatenate([seq_data, pad], axis=0)

    seq_data = seq_data[None, :, :]  # (1, 30, 63)

    pred = model(seq_data, training=False).numpy()
    class_id = int(np.argmax(pred, axis=-1)[0])
    predicted_label = GESTURES[class_id]

    if len(current_sequence) == len(PASSWORD_ORDER):
        decision = "ACCESS GRANTED" if current_sequence == PASSWORD_ORDER else "ACCESS DENIED"
    else:
        decision = "WAITING"

    result = {
        "timestamp": datetime.now().isoformat(),
        "predicted_id": class_id,
        "predicted_label": predicted_label,
        "decision": decision,
        "current_sequence": current_sequence,
    }

    with open("result.json", "w") as f:
        json.dump(result, f, indent=2)

    return jsonify(result)


@app.route("/latest_result", methods=["GET"])
def latest_result():
    try:
        with open("result.json", "r") as f:
            result = json.load(f)
        return jsonify(result)
    except Exception:
        return jsonify({"error": "No result yet"}), 404


public_url = ngrok.connect(5000)
print("NGROK URL:", public_url)

app.run(host="0.0.0.0", port=5000)

✅ Model loaded
NGROK URL: NgrokTunnel: "https://elaine-mitral-gabriel.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:23:45] "GET /latest_result HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:23:46] "GET /latest_result HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:23:47] "GET /latest_result HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:23:48] "GET /latest_result HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:23:50] "GET /latest_result HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:23:51] "GET /latest_result HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:23:52] "GET /latest_result HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Apr/2026 17:23:53] "GET /latest_result HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 